# Assignment 7 — Transfer Learning

**Aim:** Implement transfer learning using pre-trained AlexNet, VGG16, ResNet50, and EfficientNetB0 models for image classification, and compare their performance.

## 1. Import Libraries

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time

from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0

print('TensorFlow:', tf.__version__)

## 2. Load and Preprocess MNIST

MNIST images are grayscale (28×28), but ImageNet-pretrained models expect 3-channel (RGB) input. We convert grayscale to RGB by repeating across 3 channels, and resize to 32×32 for the Keras models. A smaller subset is used to keep training practical in Colab.

In [ ]:
# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

IMG_SIZE = 32
TRAIN_SAMPLES = 10000
TEST_SAMPLES = 2000

# Convert grayscale to 3-channel (RGB) by repeating
x_train = np.repeat(x_train[..., np.newaxis], 3, axis=-1)
x_test = np.repeat(x_test[..., np.newaxis], 3, axis=-1)

# Resize images and normalize pixel values to 0-1
x_train = tf.image.resize(x_train, (IMG_SIZE, IMG_SIZE)).numpy().astype('float32') / 255.0
x_test = tf.image.resize(x_test, (IMG_SIZE, IMG_SIZE)).numpy().astype('float32') / 255.0

# Take a smaller subset for faster training
x_train_small = x_train[:TRAIN_SAMPLES]
y_train_small = y_train[:TRAIN_SAMPLES]
x_test_small = x_test[:TEST_SAMPLES]
y_test_small = y_test[:TEST_SAMPLES]

print('Training:', x_train_small.shape, y_train_small.shape)
print('Testing :', x_test_small.shape, y_test_small.shape)

## Sample Images

In [ ]:
# Display a few sample images from the training set
plt.figure(figsize=(8, 3))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(x_train_small[i])
    plt.title(str(y_train_small[i]))
    plt.axis('off')
plt.tight_layout()
plt.show()

## 3. Build Transfer Learning Models (VGG16, ResNet50, EfficientNetB0)

For each model, we:
- Load the ImageNet-pretrained base model with `include_top=False`
- Freeze all convolutional layers (no training on the base)
- Add a new classification head (GlobalAveragePooling + Dense + Dropout + Softmax)

**Note:** `tf.keras.applications` does not provide an official pretrained AlexNet model, so AlexNet is handled separately using PyTorch's torchvision.

In [ ]:
# VGG16 transfer learning model
def build_vgg16():
    base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
    base.trainable = False    # Freeze pretrained layers
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')    # 10 MNIST classes
    ])
    return model

# ResNet50 transfer learning model
def build_resnet50():
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    return model

# EfficientNetB0 transfer learning model
def build_efficientnetb0():
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    return model

# Compile helper function
def compile_model(model):
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

## 4. Train and Evaluate VGG16, ResNet50, and EfficientNetB0

In [ ]:
EPOCHS = 3
BATCH_SIZE = 64
results = []
histories = {}

def train_keras_model(name, builder):
    """Train a Keras model and record results."""
    print('\n' + '=' * 60)
    print(f'Training: {name}')
    model = compile_model(builder())

    start = time.time()
    history = model.fit(
        x_train_small, y_train_small,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        verbose=1
    )
    train_time = time.time() - start

    # Evaluate on the test set
    loss, acc = model.evaluate(x_test_small, y_test_small, verbose=0)
    results.append({'Model': name, 'Test Accuracy': acc, 'Test Loss': loss, 'Training Time (s)': train_time})
    histories[name] = history
    return model

# Train all three Keras-based models
vgg16_model = train_keras_model('VGG16', build_vgg16)
resnet50_model = train_keras_model('ResNet50', build_resnet50)
efficientnet_model = train_keras_model('EfficientNetB0', build_efficientnetb0)

## 5. Pretrained AlexNet (via PyTorch)

AlexNet is loaded from `torchvision` because TensorFlow/Keras does not have an official ImageNet-pretrained AlexNet. The convolutional feature extractor is frozen and only the classifier is adapted for 10 MNIST classes.

In [ ]:
# Install PyTorch/torchvision if needed (uncomment in Colab)
# !pip -q install torch torchvision

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import alexnet, AlexNet_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch device:', device)

# Convert numpy arrays to PyTorch tensors (channels-first format for PyTorch)
xtr = torch.tensor(x_train_small).permute(0, 3, 1, 2)
ytr = torch.tensor(y_train_small, dtype=torch.long)
xte = torch.tensor(x_test_small).permute(0, 3, 1, 2)
yte = torch.tensor(y_test_small, dtype=torch.long)

train_loader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(xte, yte), batch_size=64, shuffle=False)

# Load pretrained AlexNet and freeze the feature extractor
weights = AlexNet_Weights.DEFAULT
alexnet_model = alexnet(weights=weights)
for param in alexnet_model.features.parameters():
    param.requires_grad = False    # Freeze convolutional layers

# Replace the last classifier layer for 10-class output
alexnet_model.classifier[6] = nn.Linear(alexnet_model.classifier[6].in_features, 10)
alexnet_model = alexnet_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(alexnet_model.classifier.parameters(), lr=1e-3)

alex_train_acc = []
alex_val_acc = []
start = time.time()

for epoch in range(EPOCHS):
    # Training phase
    alexnet_model.train()
    correct = total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        # Resize to 224x224 as AlexNet expects ImageNet-sized input
        xb = torch.nn.functional.interpolate(xb, size=(224, 224), mode='bilinear', align_corners=False)
        optimizer.zero_grad()
        out = alexnet_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == yb).sum().item()
        total += yb.size(0)
    train_acc = correct / total

    # Evaluation phase
    alexnet_model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            xb = torch.nn.functional.interpolate(xb, size=(224, 224), mode='bilinear', align_corners=False)
            out = alexnet_model(xb)
            correct += (out.argmax(1) == yb).sum().item()
            total += yb.size(0)
    val_acc = correct / total

    alex_train_acc.append(train_acc)
    alex_val_acc.append(val_acc)
    print(f'Epoch {epoch+1}/{EPOCHS} - train_accuracy: {train_acc:.4f} - test_accuracy: {val_acc:.4f}')

alexnet_time = time.time() - start
results.append({'Model': 'AlexNet', 'Test Accuracy': alex_val_acc[-1], 'Test Loss': np.nan, 'Training Time (s)': alexnet_time})

## 6. Compare Model Performance

In [ ]:
# Display results as a sorted table
results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False).reset_index(drop=True)
results_df

In [ ]:
# Bar chart comparing test accuracy across all models
plt.figure(figsize=(8, 5))
plt.bar(results_df['Model'], results_df['Test Accuracy'])
plt.ylim(0, 1.0)
plt.ylabel('Test Accuracy')
plt.xlabel('Model')
plt.title('Transfer Learning Model Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 7. Training Curves

In [ ]:
# Validation accuracy curves for the three Keras models
plt.figure(figsize=(9, 5))
for name, history in histories.items():
    plt.plot(history.history['val_accuracy'], marker='o', label=name)
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.title('Validation Accuracy Comparison (Keras Models)')
plt.legend()
plt.grid(True)
plt.show()

# AlexNet accuracy curve
plt.figure(figsize=(9, 5))
plt.plot(range(1, EPOCHS + 1), alex_train_acc, marker='o', label='AlexNet Train')
plt.plot(range(1, EPOCHS + 1), alex_val_acc, marker='o', label='AlexNet Test')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('AlexNet Accuracy')
plt.legend()
plt.grid(True)
plt.show()

## Conclusion

In this assignment, we:
1. Implemented transfer learning using four pretrained models: AlexNet, VGG16, ResNet50, and EfficientNetB0.
2. Froze the pretrained convolutional layers and trained only the classification heads.
3. Compared all four models based on test accuracy and training time.

**Key observations:**
- The best performing model can be identified from the results table above.
- AlexNet was implemented using PyTorch (torchvision) since Keras does not provide a pretrained AlexNet.
- VGG16, ResNet50, and EfficientNetB0 were implemented using TensorFlow/Keras.
- Since MNIST is very different from ImageNet, the comparison demonstrates the concept of transfer learning rather than reflecting which architecture is best for natural-image tasks.
- Training time varies significantly between architectures due to differences in model complexity.